In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, to_hex
import seaborn as sns
import scanpy as sc
import anndata as ad
from scipy import sparse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set scanpy settings
sc.settings.verbosity = 3  # verbosity level

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
adata = ad.read_h5ad('colon_adata_clustered.h5ad')

In [3]:
print(adata)

AnnData object with n_obs × n_vars = 424423 × 3000
    obs: 'fov', 'RNA_RNA.QC.Module_Cell.Typing.InSituType.1_1_clusters', 'RNA_RNA.QC.Module_Cell.Typing.InSituType.1_1_posterior_probability', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.G', 'Max.G', 'Mean.Membrane', 'Max.Membrane', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'SplitRatioToLocal', 'NucArea', 'NucAspectRatio', 'Circularity', 'Eccentricity', 'Perimeter', 'Solidity', 'cell_id', 'assay_type', 'version', 'Run_Tissue_name', 'Panel', 'cellSegmentationSetId', 'cellSegmentationSetName', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quan

## Grouped marker plots: tumor/cancer vs normal

Combined expression of marker groups on UMAP with cluster distribution bars (same style as Analysis III). Gene lists are shown at the bottom of each plot.

In [7]:
# Setup: colormap (same as Analysis III), paths, cluster annotation
from matplotlib.colors import LinearSegmentedColormap, Normalize

colors = ["blue", "green", "yellow", "red"]
custom_cmap = LinearSegmentedColormap.from_list("BlueGreenYellowRed", colors, N=256)

feature_plot_root = Path("feature_plots")
cluster_col = "leiden"
clusters = sorted(adata.obs[cluster_col].astype(str).unique())
genes_in_adata = set(adata.var_names)
_obs_key = "_group_combined_"  # temporary obs key for combined score in plots

### Tumor / cancerous cell markers

Gene groups for tumor and cancer-associated compartments. Each group is plotted as combined (mean) expression on UMAP with cluster bars; genes are listed at the bottom.

In [4]:
# Tumor/cancer gene groups (only genes present in adata are used for the combined score)
tumor_groups = {
    "Epithelial_malignant": [
        "TACSTD2", "CEACAM5", "REG4", "KRT8", "KRT19", "KRT20",
        "CLDN1", "CLDN4", "CEACAM7", "DMBT1"
    ],
    "Tumor_EMT_like": [
        "VIM", "ITGA5", "FN1", "ZEB1", "SNAI2", "TWIST1", "ZEB2",
        "COL1A1", "COL1A2", "TAGLN"
    ],
    "Tumor_Response_TGFB_AP1_JNK": ["TGFB2", "TGFBR2", "FOS"],
    "CAF": [
        "PDPN", "ACTA2", "TAGLN", "POSTN", "THY1", "INHBA",
        "CXCL12", "IL6", "CCL2", "PTGS2", "CXCL8", "CSF1"
    ],
    "SPP1_TAM": ["SPP1", "LGALS3", "TREM2", "GPNMB", "CTSL"],
    "Lymphoid_Stromal": ["CXCL13", "CCL19"],
}

In [12]:
# Plot each tumor group: combined expression on UMAP + cluster bars + gene list at bottom
tumor_dir = feature_plot_root / "grouped_tumor"
tumor_dir.mkdir(parents=True, exist_ok=True)

for group_name, gene_list in tumor_groups.items():
    use_genes = [g for g in gene_list if g in genes_in_adata]
    if not use_genes:
        continue
    # Combined score = sum of expression across genes (UMAP colored by this sum)
    combined = np.zeros(adata.n_obs)
    for g in use_genes:
        x = adata[:, g].X
        combined += (x.toarray().flatten() if sparse.issparse(x) else np.asarray(x).flatten())
    # No division: keep sum. Expressing = sum > 0; bar length = % of those cells per cluster.

    adata.obs[_obs_key] = combined
    expressing = combined > 0  # cells with sum of gene expression > 0
    n_exp = expressing.sum()
    if n_exp == 0:
        adata.obs.drop(columns=[_obs_key], inplace=True)
        continue

    vmax = np.percentile(combined[expressing], 99) or 1.0
    norm = Normalize(vmin=0, vmax=max(vmax, 1e-6))
    umap_colors_rgba = custom_cmap(norm(combined))

    cluster_pct = []
    cluster_mean_color = []
    for cl in clusters:
        in_cl = (adata.obs[cluster_col].astype(str) == cl).values
        cluster_pct.append((expressing & in_cl).sum() / n_exp)
        cluster_mean_color.append(
            umap_colors_rgba[in_cl].mean(axis=0) if in_cl.sum() > 0 else np.array(custom_cmap(0))
        )

    fig = plt.figure(figsize=(14, max(6, len(clusters) * 0.25) + 1.2))
    ax_umap = fig.add_axes([0.05, 0.22, 0.55, 0.72])
    ax_bars = fig.add_axes([0.65, 0.22, 0.32, 0.72])
    ax_text = fig.add_axes([0.05, 0.02, 0.93, 0.18])
    ax_text.axis("off")

    sc.pl.umap(
        adata, color=_obs_key, ax=ax_umap, cmap=custom_cmap, size=2,
        vmax="p99", show=False, title=f"{group_name} (combined)"
    )

    y_pos = np.arange(len(clusters))[::-1]
    ax_bars.barh(y_pos, np.array(cluster_pct), height=0.75, color=cluster_mean_color,
                 edgecolor="gray", linewidth=0.3)
    ax_bars.set_yticks(y_pos)
    ax_bars.set_yticklabels(clusters, fontsize=8)
    ax_bars.set_xlim(0, 1)
    ax_bars.set_xlabel("% of expressing cells in cluster")
    ax_bars.set_title("Bar color = avg UMAP color in cluster")
    sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax_bars, shrink=0.6, label="Expression scale")

    # Show markers in lines of ~10 genes so they fit
    chunks = [use_genes[i:i+10] for i in range(0, len(use_genes), 10)]
    gene_list_str = "Markers:\n" + "\n".join(", ".join(c) for c in chunks)
    ax_text.text(0.5, 0.5, gene_list_str, transform=ax_text.transAxes,
                 fontsize=7, verticalalignment="center", horizontalalignment="center",
                 bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3))

    fig.savefig(tumor_dir / f"{group_name}_combined.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    adata.obs.drop(columns=[_obs_key], inplace=True)

print(f"Saved tumor grouped plots to {tumor_dir.resolve()}")

Saved tumor grouped plots to /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/grouped_tumor


### Normal cell markers

Same layout: combined expression on UMAP, cluster bars, and gene list at bottom. Groups: epithelial (healthy), fibroblast, endothelial, SMC, pericyte.

In [5]:
# Normal cell gene groups (larger groups: epithelial, fibroblast, endothelial, SMC, pericyte)
normal_groups = {
    "Epithelial_normal": [
        "LGALS4", "SLC26A3", "SLC9A3", "AQP8", "CA1", "CA2", "FABP1",
        "GUCA2A", "MUC2", "FCGBP", "CLCA1", "AGR2", "SPINK4", "ZG16",
        "DEFA5", "DEFA6", "LYZ", "PLA2G2A", "REG3A", "CHGA", "SCG5", "NEUROD1",
        "OLFM4", "SMOC2", "GCG", "RGS2"
    ],
    "Fibroblast": [
        "VIM", "PDGFRA", "PDGFRB", "LUM", "COL1A1", "COL1A2", "COL3A1",
        "COL6A1", "COL14A1", "PI16", "COL5A1", "COL5A2", "LOX", "FBLN1", "FBLN2",
        "FN1", "VCAM1", "MGP", "SFRP2", "C3", "SPARC", "MMP1", "CXCL13", "CCL19"
    ],
    "Endothelial": [
        "PECAM1", "KDR", "ENG", "PLVAP", "PDPN", "CCL21"
    ],
    "SMC": [
        "MYH11", "ACTG2", "DES", "ACTA2", "TAGLN", "CNN1", "MYL9", "TPM2", "CALD1"
    ],
    "Pericyte": [
        "STEAP4", "PDGFRB", "MCAM", "NOTCH3", "ABCC9", "DES"
    ],
}

In [14]:
# Plot each normal group: combined expression on UMAP + cluster bars + gene list at bottom
normal_dir = feature_plot_root / "grouped_normal"
normal_dir.mkdir(parents=True, exist_ok=True)

for group_name, gene_list in normal_groups.items():
    use_genes = [g for g in gene_list if g in genes_in_adata]
    if not use_genes:
        continue
    # Combined score = sum of expression across genes
    combined = np.zeros(adata.n_obs)
    for g in use_genes:
        x = adata[:, g].X
        combined += (x.toarray().flatten() if sparse.issparse(x) else np.asarray(x).flatten())
    # Expressing = sum > 0; bar length = % of those cells in each cluster; bar color = avg UMAP color in cluster.

    adata.obs[_obs_key] = combined
    expressing = combined > 0
    n_exp = expressing.sum()
    if n_exp == 0:
        adata.obs.drop(columns=[_obs_key], inplace=True)
        continue

    vmax = np.percentile(combined[expressing], 99) or 1.0
    norm = Normalize(vmin=0, vmax=max(vmax, 1e-6))
    umap_colors_rgba = custom_cmap(norm(combined))

    cluster_pct = []
    cluster_mean_color = []
    for cl in clusters:
        in_cl = (adata.obs[cluster_col].astype(str) == cl).values
        cluster_pct.append((expressing & in_cl).sum() / n_exp)
        cluster_mean_color.append(
            umap_colors_rgba[in_cl].mean(axis=0) if in_cl.sum() > 0 else np.array(custom_cmap(0))
        )

    fig = plt.figure(figsize=(14, max(6, len(clusters) * 0.25) + 1.2))
    ax_umap = fig.add_axes([0.05, 0.22, 0.55, 0.72])
    ax_bars = fig.add_axes([0.65, 0.22, 0.32, 0.72])
    ax_text = fig.add_axes([0.05, 0.02, 0.93, 0.18])
    ax_text.axis("off")

    sc.pl.umap(
        adata, color=_obs_key, ax=ax_umap, cmap=custom_cmap, size=2,
        vmax="p99", show=False, title=f"{group_name} (combined)"
    )

    y_pos = np.arange(len(clusters))[::-1]
    ax_bars.barh(y_pos, np.array(cluster_pct), height=0.75, color=cluster_mean_color,
                 edgecolor="gray", linewidth=0.3)
    ax_bars.set_yticks(y_pos)
    ax_bars.set_yticklabels(clusters, fontsize=8)
    ax_bars.set_xlim(0, 1)
    ax_bars.set_xlabel("% of expressing cells in cluster")
    ax_bars.set_title("Bar color = avg UMAP color in cluster")
    sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax_bars, shrink=0.6, label="Expression scale")

    chunks = [use_genes[i:i+10] for i in range(0, len(use_genes), 10)]
    gene_list_str = "Markers:\n" + "\n".join(", ".join(c) for c in chunks)
    ax_text.text(0.5, 0.5, gene_list_str, transform=ax_text.transAxes,
                 fontsize=7, verticalalignment="center", horizontalalignment="center",
                 bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3))

    fig.savefig(normal_dir / f"{group_name}_combined.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    adata.obs.drop(columns=[_obs_key], inplace=True)

print(f"Saved normal grouped plots to {normal_dir.resolve()}")

Saved normal grouped plots to /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/grouped_normal


## Clusters of interest

Mark cells that belong to the confirmed tumor-related clusters so you can filter (e.g. `adata[adata.obs["in_clusters_of_interest"]]`) for targeted plotting.

In [8]:
# Set of clusters of interest (confirmed tumor-related)
clusters_of_interest = {"0", "2", "4", "6", "7", "11", "12"}

# Add boolean column: True if cell is in one of these clusters
adata.obs["clusters_of_interest"] = adata.obs[cluster_col].astype(str).isin(clusters_of_interest)
n_marked = adata.obs["clusters_of_interest"].sum()
print(f"Cells in clusters of interest: {n_marked} / {adata.n_obs} ({100*n_marked/adata.n_obs:.1f}%)")

Cells in clusters of interest: 228907 / 424423 (53.9%)


## Per-FOV tumor group enrichment and intensity

For each FOV and each tumor group, compute and store:
- `total_cells_in_fov`: total number of cells in that FOV
- `expressing_cells_in_fov`: number of cells with combined group expression > 0
- `mean_expr_in_fov_all_cells`: mean combined expression across all cells in that FOV
- `mean_expr_in_fov_expressing_cells`: mean combined expression among expressing cells only

This keeps raw values in a reusable table so percentages can be computed later as needed (`expressing_cells_in_fov / total_cells_in_fov`).

### Step 1 - Persist marker group definitions in `adata.uns`

Store tumor and normal marker dictionaries in `adata.uns` so they can be reused consistently across later analysis and plotting steps.

In [9]:
# Ensure marker group dictionaries are saved in adata.uns for reuse
adata.uns["tumor_groups"] = tumor_groups
adata.uns["normal_groups"] = normal_groups

print(f"Stored tumor groups in adata.uns['tumor_groups'] ({len(tumor_groups)} groups)")
print(f"Stored normal groups in adata.uns['normal_groups'] ({len(normal_groups)} groups)")

Stored tumor groups in adata.uns['tumor_groups'] (6 groups)
Stored normal groups in adata.uns['normal_groups'] (5 groups)


### Step 2 - Compute per-FOV tumor metrics (raw values + derived percentage)

For each tumor group, this computes per FOV:
- total cells
- expressing cells
- percentage expressing (`expressing / total * 100`)
- mean combined expression across all cells
- mean combined expression among expressing cells

It also records which marker genes were used (present in `adata.var_names`) and which were missing.

In [10]:
# Compute per-FOV tumor metrics using combined marker expression per group
fov_col = "fov"
if fov_col not in adata.obs.columns:
    raise KeyError(f"Column '{fov_col}' not found in adata.obs")

# Keep only non-null FOV labels for aggregation
fov_series = adata.obs[fov_col].astype(str)
valid_fov_mask = adata.obs[fov_col].notna().values
all_fovs = np.sort(fov_series[valid_fov_mask].unique())

records = []
missing_group_records = []

for group_name, gene_list in tumor_groups.items():
    use_genes = [g for g in gene_list if g in adata.var_names]
    missing_genes = [g for g in gene_list if g not in adata.var_names]

    if len(use_genes) == 0:
        missing_group_records.append({
            "group": group_name,
            "reason": "no marker genes present in adata.var_names",
            "n_requested_genes": len(gene_list),
        })
        continue

    # Fast group score: sum marker expression across selected genes per cell
    X_group = adata[:, use_genes].X
    combined = np.asarray(X_group.sum(axis=1)).ravel()

    temp = pd.DataFrame({
        "fov": fov_series.values,
        "combined_expr": combined,
    })
    temp = temp.loc[valid_fov_mask].copy()
    temp["is_expressing"] = temp["combined_expr"] > 0

    total_cells = temp.groupby("fov", sort=False).size().reindex(all_fovs, fill_value=0)
    expressing_cells = (
        temp.groupby("fov", sort=False)["is_expressing"]
        .sum()
        .astype(int)
        .reindex(all_fovs, fill_value=0)
    )
    sum_expr = temp.groupby("fov", sort=False)["combined_expr"].sum().reindex(all_fovs, fill_value=0.0)

    mean_expr_all_cells = (sum_expr / total_cells.replace(0, np.nan)).fillna(0.0)
    mean_expr_expressing = (sum_expr / expressing_cells.replace(0, np.nan)).fillna(0.0)
    pct_expr_in_fov = ((expressing_cells / total_cells.replace(0, np.nan)) * 100.0).fillna(0.0)

    group_df = pd.DataFrame({
        "fov": all_fovs,
        "tumor_group": group_name,
        "total_cells_in_fov": total_cells.values.astype(int),
        "expressing_cells_in_fov": expressing_cells.values.astype(int),
        "pct_expr_in_fov": pct_expr_in_fov.values,
        "mean_expr_in_fov_all_cells": mean_expr_all_cells.values,
        "mean_expr_in_fov_expressing_cells": mean_expr_expressing.values,
        "n_marker_genes_used": len(use_genes),
        "n_marker_genes_missing": len(missing_genes),
        "genes_used": ";".join(use_genes),
        "genes_missing": ";".join(missing_genes),
    })
    records.append(group_df)

if len(records) == 0:
    raise RuntimeError("No tumor groups had any genes present in adata.var_names")

fov_tumor_metrics_long = pd.concat(records, ignore_index=True)

# Optional overall ranking score for convenience (keeps raw columns untouched)
fov_tumor_metrics_long["rank_score"] = (
    fov_tumor_metrics_long["mean_expr_in_fov_expressing_cells"]
    * fov_tumor_metrics_long["expressing_cells_in_fov"]
)

# Global sorted table requested: highest expression, then highest expressing-cell count
fov_tumor_metrics_sorted = fov_tumor_metrics_long.sort_values(
    by=[
        "mean_expr_in_fov_expressing_cells",
        "expressing_cells_in_fov",
        "pct_expr_in_fov",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

# Wide table (one row per FOV) for fast plotting/filtering later
fov_tumor_metrics_wide = fov_tumor_metrics_long.pivot(
    index="fov",
    columns="tumor_group",
    values=[
        "total_cells_in_fov",
        "expressing_cells_in_fov",
        "pct_expr_in_fov",
        "mean_expr_in_fov_all_cells",
        "mean_expr_in_fov_expressing_cells",
    ],
)
fov_tumor_metrics_wide.columns = [f"{metric}__{group}" for metric, group in fov_tumor_metrics_wide.columns]
fov_tumor_metrics_wide = fov_tumor_metrics_wide.reset_index()

# Save in adata.uns
adata.uns["fov_tumor_metrics_long"] = fov_tumor_metrics_long
adata.uns["fov_tumor_metrics_sorted"] = fov_tumor_metrics_sorted
adata.uns["fov_tumor_metrics_wide"] = fov_tumor_metrics_wide
adata.uns["fov_tumor_metrics_missing_groups"] = pd.DataFrame(missing_group_records)

print(f"Computed per-FOV tumor metrics for {fov_tumor_metrics_long['tumor_group'].nunique()} groups")
print(f"Rows in long table: {len(fov_tumor_metrics_long)}")
print(f"Rows in sorted table: {len(fov_tumor_metrics_sorted)}")
print(f"Rows in wide table: {len(fov_tumor_metrics_wide)}")

Computed per-FOV tumor metrics for 6 groups
Rows in long table: 2166
Rows in sorted table: 2166
Rows in wide table: 361


### Step 3 - Export CSV files sorted by highest expression and expressing cells

This writes CSV outputs for downstream work outside the notebook:
- one global sorted long table across all FOV-group pairs
- one sorted file per tumor group
- one wide per-FOV table for quick plotting/filtering

In [11]:
# Export sorted CSV tables
tumor_fov_dir = feature_plot_root / "per_fov_tumor"
tumor_fov_dir.mkdir(parents=True, exist_ok=True)

global_sorted_csv = tumor_fov_dir / "fov_tumor_group_metrics_sorted_global.csv"
wide_csv = tumor_fov_dir / "fov_tumor_group_metrics_wide.csv"
missing_groups_csv = tumor_fov_dir / "fov_tumor_groups_missing_in_adata.csv"

fov_tumor_metrics_sorted.to_csv(global_sorted_csv, index=False)
fov_tumor_metrics_wide.to_csv(wide_csv, index=False)
pd.DataFrame(adata.uns["fov_tumor_metrics_missing_groups"]).to_csv(missing_groups_csv, index=False)

# Also save one sorted CSV per tumor group
for group_name, gdf in fov_tumor_metrics_long.groupby("tumor_group", sort=False):
    gdf_sorted = gdf.sort_values(
        by=[
            "mean_expr_in_fov_expressing_cells",
            "expressing_cells_in_fov",
            "pct_expr_in_fov",
        ],
        ascending=[False, False, False],
    )
    out_path = tumor_fov_dir / f"fov_tumor_metrics_{group_name}.csv"
    gdf_sorted.to_csv(out_path, index=False)

print(f"Saved global sorted metrics CSV: {global_sorted_csv.resolve()}")
print(f"Saved wide metrics CSV: {wide_csv.resolve()}")
print(f"Saved missing-group report CSV: {missing_groups_csv.resolve()}")
print(f"Saved per-group sorted CSV files in: {tumor_fov_dir.resolve()}")

Saved global sorted metrics CSV: /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/per_fov_tumor/fov_tumor_group_metrics_sorted_global.csv
Saved wide metrics CSV: /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/per_fov_tumor/fov_tumor_group_metrics_wide.csv
Saved missing-group report CSV: /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/per_fov_tumor/fov_tumor_groups_missing_in_adata.csv
Saved per-group sorted CSV files in: /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/per_fov_tumor


### Step 4 - Lightweight FOV summary plots (top N + all 361)

These plots use precomputed per-FOV metrics from `adata.uns['fov_tumor_metrics_long']`, so they are fast to generate.

- Plot A: top `TOP_N` FOVs for each tumor group
- Plot B: top `TOP_N` FOVs by average across tumor groups
- Plot C: all FOVs (ranked) using average across tumor groups

In [18]:
# Configurable plotting parameters
TOP_N = 50  # change this to any top-N you want
PLOT_DIR = feature_plot_root / "per_fov_tumor"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if "fov_tumor_metrics_long" not in adata.uns:
    raise KeyError("Run the per-FOV metric computation step first to create adata.uns['fov_tumor_metrics_long']")

plot_df = adata.uns["fov_tumor_metrics_long"].copy()

required_cols = [
    "fov",
    "tumor_group",
    "total_cells_in_fov",
    "expressing_cells_in_fov",
    "pct_expr_in_fov",
    "mean_expr_in_fov_expressing_cells",
]
missing_cols = [c for c in required_cols if c not in plot_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in fov_tumor_metrics_long: {missing_cols}")

# Average-across-groups table (one row per FOV)
fov_avg = (
    plot_df.groupby("fov", as_index=False)
    .agg(
        avg_total_cells_in_fov=("total_cells_in_fov", "mean"),
        avg_expressing_cells_in_fov=("expressing_cells_in_fov", "mean"),
        avg_pct_expr_in_fov=("pct_expr_in_fov", "mean"),
        avg_mean_expr_in_fov_expressing_cells=("mean_expr_in_fov_expressing_cells", "mean"),
    )
)

# Ranking score for ordering (can be changed if needed)
fov_avg["avg_rank_score"] = (
    fov_avg["avg_mean_expr_in_fov_expressing_cells"] * fov_avg["avg_expressing_cells_in_fov"]
)

print(f"Top-N set to: {TOP_N}")
print(f"Tumor groups: {plot_df['tumor_group'].nunique()}")
print(f"Total FOVs in summary: {fov_avg['fov'].nunique()}")

Top-N set to: 50
Tumor groups: 6
Total FOVs in summary: 361


In [ ]:
# Plot A: Top-N FOVs per tumor group (one image per group, bar plot + hue colorbar)
group_order = sorted(plot_df["tumor_group"].unique())

for g in group_order:
    gdf = plot_df[plot_df["tumor_group"] == g].copy()
    gdf = gdf.sort_values(
        by=["mean_expr_in_fov_expressing_cells", "expressing_cells_in_fov", "pct_expr_in_fov"],
        ascending=[False, False, False],
    ).head(TOP_N)

    # Plot smallest at top, largest at bottom for readability
    gdf_plot = gdf.sort_values("mean_expr_in_fov_expressing_cells", ascending=True)
    pct_vals = gdf_plot["pct_expr_in_fov"].to_numpy(dtype=float)
    pct_max = max(pct_vals.max(), 1e-9)
    norm = Normalize(vmin=0, vmax=pct_max)

    fig, ax = plt.subplots(figsize=(10, max(6, TOP_N * 0.22)))
    ax.barh(
        gdf_plot["fov"].astype(str),
        gdf_plot["mean_expr_in_fov_expressing_cells"],
        color=custom_cmap(norm(pct_vals)),
        edgecolor="black",
        linewidth=0.25,
    )

    ax.set_title(f"{g} | Top {TOP_N} FOVs")
    ax.set_xlabel("Mean expression (expressing cells)")
    ax.set_ylabel("FOV")
    ax.tick_params(axis="y", labelsize=7)

    sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.015)
    cbar.set_label("% expressing in FOV")

    fig.tight_layout()
    out_a = PLOT_DIR / f"fov_top{TOP_N}_{g}_bar.png"
    fig.savefig(out_a, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Saved Plot A ({g}): {out_a.resolve()}")

In [ ]:
# Plot B: Top-N FOVs by average across all tumor groups (single bar plot + hue colorbar)
top_avg = fov_avg.sort_values(
    by=["avg_mean_expr_in_fov_expressing_cells", "avg_expressing_cells_in_fov", "avg_pct_expr_in_fov"],
    ascending=[False, False, False],
).head(TOP_N).copy()

top_avg_plot = top_avg.sort_values("avg_mean_expr_in_fov_expressing_cells", ascending=True)
avg_pct_vals = top_avg_plot["avg_pct_expr_in_fov"].to_numpy(dtype=float)
avg_pct_max = max(avg_pct_vals.max(), 1e-9)
norm = Normalize(vmin=0, vmax=avg_pct_max)

fig, ax = plt.subplots(figsize=(10, max(6, TOP_N * 0.22)))
ax.barh(
    top_avg_plot["fov"].astype(str),
    top_avg_plot["avg_mean_expr_in_fov_expressing_cells"],
    color=custom_cmap(norm(avg_pct_vals)),
    edgecolor="black",
    linewidth=0.25,
)

ax.set_title(f"Top {TOP_N} FOVs by average across tumor groups")
ax.set_xlabel("Average mean expression (expressing cells) across groups")
ax.set_ylabel("FOV")
ax.tick_params(axis="y", labelsize=7)

sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.015)
cbar.set_label("Average % expressing across groups")

fig.tight_layout()
out_b = PLOT_DIR / f"fov_top{TOP_N}_average_across_tumor_groups_bar.png"
fig.savefig(out_b, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved Plot B: {out_b.resolve()}")

### Step 5 - FOV overlays for selected top FOVs (spatial + polygons, tumor expression hue)

Build FOV plots for FOVs that appear in either:
- top `TOP_N` within at least one tumor group, or
- top `TOP_N` by average across tumor groups

Each output image has two panels:
- left: spatial scatter (`obsm['spatial']`)
- right: polygon overlay (`uns['polygons']`)

Tumor cells are colored by per-cell tumor expression score, and a side colorbar shows the hue scale.
Lower color is gray (not blue).

In [26]:
# Select FOVs from top-N per group OR top-N overall average
if "plot_df" not in globals() or "fov_avg" not in globals():
    raise KeyError("Run the plotting prep cell first (the cell that defines plot_df and fov_avg).")

# Colormap for FOV expression overlays: low is gray, then green -> yellow -> red
fov_expr_colors = ["#b3b3b3", "#4daf4a", "#ffd92f", "#e41a1c"]
fov_expr_cmap = LinearSegmentedColormap.from_list("GrayGreenYellowRed", fov_expr_colors, N=256)

top_per_group = (
    plot_df.sort_values(
        by=["tumor_group", "mean_expr_in_fov_expressing_cells", "expressing_cells_in_fov", "pct_expr_in_fov"],
        ascending=[True, False, False, False],
    )
    .groupby("tumor_group", group_keys=False)
    .head(TOP_N)
)

top_overall = fov_avg.sort_values(
    by=["avg_mean_expr_in_fov_expressing_cells", "avg_expressing_cells_in_fov", "avg_pct_expr_in_fov"],
    ascending=[False, False, False],
).head(TOP_N)

fovs_top_per_group = set(top_per_group["fov"].astype(str))
fovs_top_overall = set(top_overall["fov"].astype(str))
selected_fovs = sorted(fovs_top_per_group.union(fovs_top_overall), key=lambda x: (len(x), x))

print(f"TOP_N = {TOP_N}")
print(f"FOVs from top-per-group: {len(fovs_top_per_group)}")
print(f"FOVs from top-overall: {len(fovs_top_overall)}")
print(f"Union selected FOVs: {len(selected_fovs)}")

TOP_N = 50
FOVs from top-per-group: 173
FOVs from top-overall: 50
Union selected FOVs: 174


In [34]:
# Build per-cell tumor score and generate selected-FOV overlays (scatter + polygons)
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

poly = adata.uns.get("polygons")
if poly is None:
    raise ValueError("adata.uns['polygons'] is missing; cannot draw polygon panel.")

overlay_dir = PLOT_DIR / f"selected_fov_overlays_top{TOP_N}"
overlay_dir.mkdir(parents=True, exist_ok=True)

# Use all unique tumor marker genes present in adata
all_tumor_genes = sorted({g for genes in tumor_groups.values() for g in genes if g in adata.var_names})
if len(all_tumor_genes) == 0:
    raise RuntimeError("No tumor marker genes from tumor_groups were found in adata.var_names.")

# Per-cell tumor score = sum across all tumor marker genes
X_tumor = adata[:, all_tumor_genes].X
tumor_score_all = np.asarray(X_tumor.sum(axis=1)).ravel()

# Tumor-cell mask (reuse if already defined, otherwise infer from clusters_of_interest)
if "clusters_of_interest" in adata.obs.columns:
    tumor_mask_all = adata.obs["clusters_of_interest"].astype(bool).values
elif "clusters_of_interest" in globals():
    tumor_mask_all = adata.obs[cluster_col].astype(str).isin(clusters_of_interest).values
    adata.obs["clusters_of_interest"] = tumor_mask_all
else:
    raise KeyError("Tumor-cell mask not found. Need adata.obs['clusters_of_interest'] or clusters_of_interest set.")

# Global normalization across tumor cells for consistent color scale across FOV plots
tumor_scores_only = tumor_score_all[tumor_mask_all]
if tumor_scores_only.size == 0:
    raise RuntimeError("No tumor cells found in mask; cannot color tumor expression.")
vmax_global = np.percentile(tumor_scores_only, 99)
vmax_global = max(float(vmax_global), 1e-9)
norm_global = Normalize(vmin=0, vmax=vmax_global)

saved = []
for fov in selected_fovs:
    in_fov = adata.obs[fov_col].astype(str).values == str(fov)
    idx_fov = np.where(in_fov)[0]
    if idx_fov.size == 0:
        continue

    xy = adata.obsm["spatial"][idx_fov]
    scores_fov = tumor_score_all[idx_fov]
    tumor_mask_fov = tumor_mask_all[idx_fov]

    # Colors: non-tumor cells in light gray, tumor cells in expression hue
    point_colors = np.full((idx_fov.size, 4), fill_value=(0.85, 0.85, 0.85, 0.65), dtype=float)
    if tumor_mask_fov.any():
        point_colors[tumor_mask_fov] = fov_expr_cmap(norm_global(scores_fov[tumor_mask_fov]))

    # Lookup from cell id/index to expression color for polygon drawing
    obs_names_fov = adata.obs_names[idx_fov]
    color_lookup = {obs_name: point_colors[i] for i, obs_name in enumerate(obs_names_fov)}

    fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)

    # Left panel: spatial scatter
    axes[0].scatter(xy[:, 0], xy[:, 1], s=5, c=point_colors, linewidths=0)
    axes[0].set_title(f"FOV {fov} — spatial tumor-expression")
    axes[0].set_xlabel("CenterX_local_px")
    axes[0].set_ylabel("CenterY_local_px")
    axes[0].invert_yaxis()
    axes[0].set_aspect("equal", adjustable="box")

    # Right panel: polygons with same expression hue
    axes[1].set_title(f"FOV {fov} — polygons tumor-expression")
    axes[1].set_xlabel("x_local_px")
    axes[1].set_ylabel("y_local_px")

    poly_fov = poly[poly["fov"].astype(str) == str(fov)]
    if len(poly_fov) > 0:
        for cell_label, df_cell in poly_fov.groupby("cell"):
            col = color_lookup.get(str(cell_label), (0.8, 0.8, 0.8, 0.6))
            axes[1].plot(df_cell["x_local_px"], df_cell["y_local_px"], color=col, linewidth=0.5)
    else:
        axes[1].text(0.5, 0.5, "No polygon rows for this FOV", ha="center", va="center")

    axes[1].invert_yaxis()
    axes[1].set_aspect("equal", adjustable="box")

    fig.suptitle(
        f"FOV {fov} | colored tumor cells = clusters_of_interest; low color = gray",
        fontsize=12,
    )

    # Reserve a wider right margin, then place colorbar farther right in a custom axis
    fig.tight_layout(rect=[0, 0, 0.78, 1])
    cax = fig.add_axes([0.80, 0.18, 0.018, 0.64])
    sm = ScalarMappable(cmap=fov_expr_cmap, norm=norm_global)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Tumor expression score (sum of tumor marker genes)")

    out_path = overlay_dir / f"fov_{str(fov)}_tumor_expr_overlay.png"
    fig.savefig(out_path, dpi=240, bbox_inches="tight")
    plt.close(fig)
    saved.append(out_path)

print(f"Saved {len(saved)} selected-FOV tumor-expression overlays to {overlay_dir.resolve()}")
print(f"Global expression color scale: vmin=0, vmax(p99 tumor cells)={vmax_global:.4f}")

Saved 174 selected-FOV tumor-expression overlays to /Users/brunondibambwayeroy/Documents/Research/YALE DATA/feature_plots/per_fov_tumor/selected_fov_overlays_top50
Global expression color scale: vmin=0, vmax(p99 tumor cells)=27.9662


### Step 6 - Optional: persist updated AnnData to disk

Run this only when you want the new `adata.uns` content saved to a file for future sessions.

In [28]:
# Optional persistence step
output_h5ad = "colon_adata_clustered.h5ad"
adata.write_h5ad(output_h5ad)
print(f"Saved updated AnnData to: {Path(output_h5ad).resolve()}")

Saved updated AnnData to: /Users/brunondibambwayeroy/Documents/Research/YALE DATA/colon_adata_clustered.h5ad
